# Day 1 — Exploratory Data Analysis
## Personalized Learning Platform for Data Science & ML Engineering

**Goal:** Understand the synthetic learner dataset, validate the data generation logic, uncover key patterns in skill gaps, engagement, and dropout risk, and set up the analytical foundation for model building.

---

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Consistent dark-mode style ──
PALETTE = {
    'bg':      '#0f1117',
    'surface': '#1a1d27',
    'accent1': '#6c63ff',
    'accent2': '#00d4aa',
    'accent3': '#ff6b6b',
    'accent4': '#ffd166',
    'text':    '#e8e8f0',
    'grid':    '#2a2d3e',
}

plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['surface'],
    'axes.edgecolor':    PALETTE['grid'],
    'axes.labelcolor':   PALETTE['text'],
    'xtick.color':       PALETTE['text'],
    'ytick.color':       PALETTE['text'],
    'text.color':        PALETTE['text'],
    'grid.color':        PALETTE['grid'],
    'axes.grid':         True,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'figure.dpi':        110,
})

print('Libraries loaded successfully.')

In [ ]:
# ── Load raw dataset ──
RAW_PATH = '../data/raw/learners.csv'
df = pd.read_csv(RAW_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head()

---
## 1. Summary Statistics & Data Quality

In [ ]:
print('=== NUMERIC SUMMARY ===')
display(df.describe().round(2))

In [ ]:
print('=== MISSING VALUE ANALYSIS ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['surface'])

bars = ax.barh(missing_df.index, missing_df['Missing %'], color=PALETTE['accent3'], edgecolor='none', height=0.6)
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=9, color=PALETTE['text'])

ax.set_xlabel('Missing Value %')
ax.set_title('Missing Value Analysis by Column')
plt.tight_layout()
plt.show()

print(f'\nTotal missing cells: {missing.sum():,} / {df.size:,} ({missing.sum()/df.size*100:.2f}%)')
display(missing_df)

In [ ]:
print('=== CATEGORICAL DISTRIBUTIONS ===')
cat_cols = ['career_goal', 'current_skill_level', 'preferred_learning_pace']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor(PALETTE['bg'])
colors = [PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent4']]

for ax, col, color in zip(axes, cat_cols, colors):
    ax.set_facecolor(PALETTE['surface'])
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=color, alpha=0.85, edgecolor='none')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val}', ha='center', va='bottom', fontsize=9)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)

plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.02, color=PALETTE['text'])
plt.tight_layout()
plt.show()

---
## 2. Topic Score Distributions

In [ ]:
topic_cols = ['score_python', 'score_statistics', 'score_ml_algorithms',
              'score_deep_learning', 'score_sql', 'score_data_visualization']

topic_labels = ['Python', 'Statistics', 'ML Algorithms', 'Deep Learning', 'SQL', 'Data Viz']
topic_colors = [PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent3'],
                PALETTE['accent4'], '#a29bfe', '#fd79a8']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor(PALETTE['bg'])
fig.suptitle('Topic Score Distributions', fontsize=15, color=PALETTE['text'], y=1.01)

for ax, col, label, color in zip(axes.flat, topic_cols, topic_labels, topic_colors):
    ax.set_facecolor(PALETTE['surface'])
    data = df[col].dropna()
    ax.hist(data, bins=30, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(data.mean(), color='white', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color=PALETTE['accent3'], linestyle=':', linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(label)
    ax.set_xlabel('Score (0-100)')
    ax.legend(fontsize=8, facecolor=PALETTE['surface'], edgecolor=PALETTE['grid'])

plt.tight_layout()
plt.show()

---
## 3. Correlation Heatmap

In [ ]:
corr_cols = topic_cols + [
    'avg_quiz_score', 'avg_coding_challenge_score',
    'time_spent_hours_per_week', 'completed_modules_count',
    'prior_experience_months', 'dropout_risk'
]

corr_labels = topic_labels + [
    'Quiz Score', 'Coding Score', 'Hours/Week',
    'Modules Done', 'Experience (mo)', 'Dropout Risk'
]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['surface'])

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    mask=mask,
    xticklabels=corr_labels,
    yticklabels=corr_labels,
    linewidths=0.5,
    linecolor=PALETTE['grid'],
    ax=ax,
    cbar_kws={'shrink': 0.8},
    annot_kws={'size': 8},
)
ax.set_title('Correlation Heatmap — Topic Scores, Engagement Metrics, Dropout Risk', pad=15)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

print('Top correlations with dropout_risk:')
display(corr_matrix['dropout_risk'].drop('dropout_risk').sort_values().rename('Correlation'))

---
## 4. Target Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(PALETTE['bg'])
fig.suptitle('Target Variable Class Balance', fontsize=14, color=PALETTE['text'])

# next_best_module
ax = axes[0]
ax.set_facecolor(PALETTE['surface'])
module_counts = df['next_best_module'].value_counts()
colors_modules = [PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent3'],
                  PALETTE['accent4'], '#a29bfe', '#fd79a8', '#55efc4']
bars = ax.barh(module_counts.index, module_counts.values, color=colors_modules[:len(module_counts)], edgecolor='none', height=0.6)
for bar, val in zip(bars, module_counts.values):
    pct = val / len(df) * 100
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val} ({pct:.1f}%)',
            va='center', fontsize=9, color=PALETTE['text'])
ax.set_title('next_best_module Distribution')
ax.set_xlabel('Count')

# dropout_risk
ax = axes[1]
ax.set_facecolor(PALETTE['surface'])
dropout_counts = df['dropout_risk'].value_counts()
labels_d = ['No Risk (0)', 'At Risk (1)']
colors_d  = [PALETTE['accent2'], PALETTE['accent3']]
wedges, texts, autotexts = ax.pie(
    dropout_counts.values,
    labels=[f'{l}\n{v} ({v/len(df)*100:.1f}%)' for l, v in zip(labels_d, dropout_counts.values)],
    colors=colors_d,
    autopct='',
    startangle=90,
    wedgeprops={'edgecolor': PALETTE['bg'], 'linewidth': 2},
    textprops={'color': PALETTE['text'], 'fontsize': 11},
)
ax.set_title('dropout_risk Distribution')

plt.tight_layout()
plt.show()

print('Class imbalance ratio (dropout):', round(dropout_counts[0]/dropout_counts[1], 2), ':1')

---
## 5. Skill Gap Analysis by Career Goal & Skill Level

In [ ]:
# Average topic scores by career_goal
career_topic_avg = df.groupby('career_goal')[topic_cols].mean()

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['surface'])

x = np.arange(len(topic_cols))
width = 0.2
goal_colors = [PALETTE['accent1'], PALETTE['accent2'], PALETTE['accent3'], PALETTE['accent4']]

for i, (goal, color) in enumerate(zip(career_topic_avg.index, goal_colors)):
    offsets = x + i * width - 1.5 * width
    bars = ax.bar(offsets, career_topic_avg.loc[goal], width=width, label=goal,
                  color=color, alpha=0.85, edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(topic_labels, rotation=15, ha='right')
ax.set_ylabel('Average Score')
ax.set_title('Average Topic Scores by Career Goal')
ax.legend(facecolor=PALETTE['surface'], edgecolor=PALETTE['grid'])
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# Average topic scores by current_skill_level
level_topic_avg = df.groupby('current_skill_level')[topic_cols].mean().reindex(['Beginner', 'Intermediate', 'Advanced'])

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['surface'])

x = np.arange(len(topic_cols))
width = 0.25
level_colors = [PALETTE['accent3'], PALETTE['accent4'], PALETTE['accent2']]

for i, (level, color) in enumerate(zip(level_topic_avg.index, level_colors)):
    offsets = x + i * width - width
    ax.bar(offsets, level_topic_avg.loc[level], width=width, label=level,
           color=color, alpha=0.85, edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(topic_labels, rotation=15, ha='right')
ax.set_ylabel('Average Score')
ax.set_title('Average Topic Scores by Skill Level')
ax.legend(facecolor=PALETTE['surface'], edgecolor=PALETTE['grid'])
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# Dropout risk vs engagement metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(PALETTE['bg'])
fig.suptitle('Dropout Risk vs Engagement Metrics', fontsize=14, color=PALETTE['text'])

engagement_cols = ['avg_quiz_score', 'time_spent_hours_per_week', 'completed_modules_count']
eng_labels = ['Avg Quiz Score', 'Hours / Week', 'Modules Completed']

for ax, col, label in zip(axes, engagement_cols, eng_labels):
    ax.set_facecolor(PALETTE['surface'])
    for risk_val, color, name in [(0, PALETTE['accent2'], 'No Risk'), (1, PALETTE['accent3'], 'At Risk')]:
        data = df[df['dropout_risk'] == risk_val][col].dropna()
        ax.hist(data, bins=25, alpha=0.65, color=color, label=name, edgecolor='none')
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.legend(fontsize=9, facecolor=PALETTE['surface'], edgecolor=PALETTE['grid'])

plt.tight_layout()
plt.show()

In [ ]:
# next_best_module distribution per career goal
cross = pd.crosstab(df['next_best_module'], df['career_goal'])

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(PALETTE['bg'])
ax.set_facecolor(PALETTE['surface'])

cross.plot(kind='bar', ax=ax, color=goal_colors, alpha=0.85, edgecolor='none')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
ax.set_title('Next Best Module Recommendations by Career Goal')
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.legend(title='Career Goal', facecolor=PALETTE['surface'], edgecolor=PALETTE['grid'])

plt.tight_layout()
plt.show()

---
## 6. Outlier Analysis

In [ ]:
outlier_cols = ['prior_experience_months', 'time_spent_hours_per_week', 'completed_modules_count']
outlier_labels = ['Prior Experience (months)', 'Hours / Week', 'Modules Completed']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(PALETTE['bg'])
fig.suptitle('Outlier Detection — Box Plots (Engineered Outliers Visible)', fontsize=13, color=PALETTE['text'])

for ax, col, label in zip(axes, outlier_cols, outlier_labels):
    ax.set_facecolor(PALETTE['surface'])
    bp = ax.boxplot(
        df[col].dropna(),
        vert=True,
        patch_artist=True,
        medianprops={'color': PALETTE['accent4'], 'linewidth': 2},
        boxprops={'facecolor': PALETTE['accent1'], 'alpha': 0.5},
        whiskerprops={'color': PALETTE['text']},
        capprops={'color': PALETTE['text']},
        flierprops={'marker': 'o', 'color': PALETTE['accent3'],
                    'markerfacecolor': PALETTE['accent3'], 'markersize': 5, 'alpha': 0.7},
    )
    ax.set_title(label)
    ax.set_ylabel('Value')
    ax.set_xticks([])

plt.tight_layout()
plt.show()

---
## 7. EDA Summary & Key Findings

| Observation | Finding |
|---|---|
| **Missing values** | ~4-5% per eligible column — within target; imputation needed |
| **Outliers** | `prior_experience_months` and `time_spent_hours_per_week` have deliberate outliers (~2%) |
| **Topic scores** | Right-skewed distributions; Beginners cluster at low scores, Advanced at high |
| **Dropout risk** | Negatively correlated with quiz scores, hours/week, modules completed — validates synthetic logic |
| **Class balance** | `next_best_module` has 7 classes with moderate imbalance; macro F1 is appropriate metric |
| **Career-goal skill gaps** | ML Engineers need deep_learning strength; Data Analysts need SQL; validated by score patterns |
| **Engagement-dropout link** | Clear separation in quiz score and hours distributions between at-risk and non-risk learners |
